# Train a Drone Fire Detection Model (YOLO26)

<a href="https://colab.research.google.com/github/jakkzz/Fire-Detection-Drone/blob/main/drone_fire_detection_yolo26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Trains a fire-detection model on aerial/drone imagery using
[Ultralytics YOLO26](https://docs.ultralytics.com/models/yolo26/).

**Runtime:** `Runtime` → `Change runtime type` → **T4 GPU** (or better), then `Save`.

Once trained, use `Supervision_Image_Inferencing.ipynb` and
`Supervision_Video_Inferencing.ipynb` to run the resulting `best.pt` on images and video.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install

Version floors rather than exact pins: they guarantee the APIs this notebook uses
while still letting pip resolve against whatever PyTorch build Colab ships.

In [ ]:
%pip install -q "ultralytics>=8.4.122" "supervision>=0.30.0"

import ultralytics
ultralytics.checks()

In [ ]:
import os
from pathlib import Path

from ultralytics import YOLO

HOME = Path.cwd()
print("HOME:", HOME)

## 3. Download the dataset from Roboflow

Needs a free Roboflow API key (<https://app.roboflow.com/settings/api>).

In Colab, store it once under 🔑 **Secrets** in the left sidebar as
`ROBOFLOW_API_KEY` with *Notebook access* enabled. The cell below reads it from
there, falls back to the environment, and only then prompts — so the key never
gets committed into the notebook.

In [ ]:
%pip install -q "roboflow>=1.4.1"

In [ ]:
import getpass
import os


def get_roboflow_api_key() -> str:
    try:
        from google.colab import userdata  # type: ignore

        key = userdata.get("ROBOFLOW_API_KEY")
        if key:
            return key
    except Exception:
        pass
    key = os.environ.get("ROBOFLOW_API_KEY")
    if key:
        return key
    return getpass.getpass("Roboflow API key: ")


ROBOFLOW_API_KEY = get_roboflow_api_key()

In [ ]:
from roboflow import Roboflow

DATASETS_DIR = HOME / "datasets"
DATASETS_DIR.mkdir(exist_ok=True)
os.chdir(DATASETS_DIR)

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("tim-4ijf0").project("drone-fire-detection-byija")

# "yolov8" names the *dataset export layout* (images + YOLO .txt labels + data.yaml),
# not a model version. Roboflow has no "yolo26" export format, and it does not need one:
# this is the plain YOLO layout that YOLO26 trains on unchanged. The exported data.yaml
# uses relative "../train/images" paths, which Ultralytics resolves against the
# data.yaml's own folder, so no path munging is needed on our side.
dataset = project.version(1).download("yolov8")

os.chdir(HOME)

DATA_YAML = Path(dataset.location) / "data.yaml"
print("data.yaml:", DATA_YAML)
print(DATA_YAML.read_text())

## 4. Train

YOLO26 is end-to-end / NMS-free, so there is no `iou` NMS threshold to tune —
the model emits final boxes directly.

`results.save_dir` is captured so later cells never hardcode `runs/detect/train`;
Ultralytics increments that to `train2`, `train3`, … on repeat runs.

Drop `imgsz` to 640, or `model` to `yolo26n.pt`, if you hit an out-of-memory error.

In [ ]:
MODEL_ARCH = "yolo26m.pt"  # n / s / m / l / x

model = YOLO(MODEL_ARCH)

train_results = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=800,
    plots=True,
)

RUN_DIR = Path(train_results.save_dir)
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
print("run dir:", RUN_DIR)
print("best weights:", BEST_WEIGHTS)

In [ ]:
from IPython.display import Image, display

for artifact in ["confusion_matrix.png", "results.png", "val_batch0_pred.jpg"]:
    path = RUN_DIR / artifact
    if path.exists():
        print(artifact)
        display(Image(filename=str(path), width=700))
    else:
        print("missing:", artifact)

## 5. Validate

In [ ]:
best_model = YOLO(str(BEST_WEIGHTS))
metrics = best_model.val(data=str(DATA_YAML))

print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP75:    {metrics.box.map75:.4f}")

## 6. Predict on the test split

In [ ]:
predict_results = best_model.predict(
    source=str(Path(dataset.location) / "test" / "images"),
    conf=0.25,
    save=True,
)

PREDICT_DIR = Path(predict_results[0].save_dir)
print("predictions:", PREDICT_DIR)

In [ ]:
import glob

for image_path in sorted(glob.glob(f"{PREDICT_DIR}/*.jpg"))[:3]:
    display(Image(filename=image_path, width=700))

## 7. Export the weights

Downloads `best.pt` so you can feed it to the two inference notebooks.

In [ ]:
try:
    from google.colab import files  # type: ignore

    files.download(str(BEST_WEIGHTS))
except ImportError:
    print("Not running in Colab. Weights are at:", BEST_WEIGHTS)